# Heart Failure Mortality Prediction

**Course:** CS316 - Introduction to AI and Data Science (Term 261)

**Goal:** Predict whether a patient with heart failure died during the follow-up period.

**Dataset:** Heart Failure Clinical Records (Chicco & Jurman, 2020), from the UCI Machine Learning Repository.
It has **299 patients** and **12 clinical features**. The target is `DEATH_EVENT` (1 = died, 0 = survived).

## 1. Load the libraries

In [ ]:
# Import all the libraries we need
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier

from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                             f1_score, confusion_matrix, roc_auc_score)

## 2. Load the dataset

In [ ]:
# Read the CSV file
data = pd.read_csv("heart_failure_clinical_records.csv")

print("Dataset shape (rows, columns):", data.shape)
data.head()

## 3. Quick data check

We check three things: missing values, duplicate rows, and the balance of the target.

In [ ]:
# Missing values
print("Missing values per column:")
print(data.isnull().sum())

# Duplicate rows
print("\nDuplicate rows:", data.duplicated().sum())

# Target balance (how many died vs survived)
print("\nTarget (DEATH_EVENT) distribution:")
print(data["DEATH_EVENT"].value_counts(normalize=True).round(3))

## 4. Features and target

We separate the features (X) from the target (y), then split the data into train and test sets.
The split is **stratified** so both sets keep the same class ratio.

In [ ]:
# Features = all columns except the target
FEATURES = [c for c in data.columns if c != "DEATH_EVENT"]
X = data[FEATURES]
y = data["DEATH_EVENT"]

# 80% for training, 20% for testing (stratified)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
print("Train size:", len(X_train), "| Test size:", len(X_test))

## 5. Train the models

We use five classifiers. Logistic Regression and KNN need scaling,
so we wrap them in a pipeline with `StandardScaler`.

In [ ]:
models = {
    "Logistic Regression": make_pipeline(
        StandardScaler(), LogisticRegression(max_iter=3000, random_state=42)
    ),
    "KNN": make_pipeline(
        StandardScaler(), KNeighborsClassifier(n_neighbors=7)
    ),
    "Decision Tree": DecisionTreeClassifier(max_depth=4, random_state=42),
    "Random Forest": RandomForestClassifier(n_estimators=500, random_state=42, n_jobs=-1),
}

# XGBoost is optional but strong (runs only if installed)
try:
    import xgboost as xgb
    models["XGBoost"] = xgb.XGBClassifier(
        n_estimators=400, learning_rate=0.05, max_depth=4,
        subsample=0.8, colsample_bytree=0.8, tree_method="hist",
        random_state=42, eval_metric="logloss"
    )
except ImportError:
    print("xgboost not installed, skipping it.")

## 6. Evaluate the models

For each model we print accuracy, precision, recall, specificity, F1 and AUC.

In [ ]:
results = []

for name, model in models.items():
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    y_prob = model.predict_proba(X_test)[:, 1]

    acc = accuracy_score(y_test, y_pred)
    prec = precision_score(y_test, y_pred)
    rec = recall_score(y_test, y_pred)
    spec = recall_score(y_test, y_pred, pos_label=0)  # specificity
    f1 = f1_score(y_test, y_pred)
    auc = roc_auc_score(y_test, y_prob)

    results.append((name, acc, prec, rec, spec, f1, auc))

    print("=== " + name + " ===")
    print("  Accuracy    : %.2f%%" % (acc * 100))
    print("  Precision   : %.2f%%" % (prec * 100))
    print("  Recall      : %.2f%%" % (rec * 100))
    print("  Specificity : %.2f%%" % (spec * 100))
    print("  F1 score    : %.3f" % f1)
    print("  AUC-ROC     : %.3f" % auc)
    print("  Confusion matrix [[TN, FP], [FN, TP]]:")
    print(" ", confusion_matrix(y_test, y_pred))
    print()

## 7. Summary table

All results in one table, sorted by accuracy.

In [ ]:
summary = pd.DataFrame(
    results,
    columns=["Model", "Accuracy", "Precision", "Recall", "Specificity", "F1", "AUC"],
).sort_values("Accuracy", ascending=False).reset_index(drop=True)

summary.round(3)

## 8. Cross-validation of the best model

A single train/test split can be lucky. 5-fold cross-validation gives a more reliable number.

In [ ]:
best_name = summary.iloc[0]["Model"]
best_model = models[best_name]

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
cv_acc = cross_val_score(best_model, X, y, cv=cv, scoring="accuracy", n_jobs=-1)

print("Best model:", best_name)
print("5-fold CV accuracy: %.2f%% +/- %.2f%%" % (cv_acc.mean() * 100, cv_acc.std() * 100))

## 9. Note about the "time" feature

The `time` feature is the follow-up period (in days). It is a strong predictor,
because patients who died have a shorter follow-up. If we remove it, accuracy drops a lot.
This is discussed as a limitation in the report.

In [ ]:
# Remove the "time" feature and re-check with Random Forest
X_notime = X.drop(columns=["time"])

rf_no_time = RandomForestClassifier(n_estimators=500, random_state=42, n_jobs=-1)
cv_acc_nt = cross_val_score(rf_no_time, X_notime, y, cv=cv, scoring="accuracy", n_jobs=-1)

print("Random Forest WITHOUT 'time' CV accuracy: %.2f%%" % (cv_acc_nt.mean() * 100))
print("Random Forest WITH 'time' CV accuracy: %.2f%%" % (cv_acc.mean() * 100))

## Conclusion

- **XGBoost** is the best model with about **83.6%** cross-validation accuracy.
- Logistic Regression and Random Forest also reach about **82%**, which is above the 80% target.
- The `time` feature carries most of the predictive signal (discussed as a limitation).